### Avoid Shuffles Module — Usage Examples
O módulo `avoid_shuffles` ajuda você a evitar shuffles desnecessários — um dos maiores vilões de performance em pipelines Spark.
Essas funções são especialmente úteis em cenários de seguros, onde datasets de apólices e sinistros podem ser enormes.

### Import Libraries
-----

In [ ]:
from pyspark.sql import SparkSession
from pyspark_utils.cleaning import (
    column_names,
    nulls,
    deduplication,
    dates,
    type_casting,
    text_cleaning
)

### Example Dataset
---

In [ ]:
claims = [
    ("POL123", "2024-01-10", 500.0, "collision"),
    ("POL123", "2024-02-15", 1200.0, "glass"),
    ("POL456", "2024-01-20", 300.0, "theft"),
    ("POL789", "2024-03-01", 900.0, "collision"),
]
columns = ["policy_number", "claim_date", "claim_amount", "claim_type"]

df = spark.createDataFrame(claims, columns)


### Selecionar apenas colunas necessárias antes do join

In [ ]:
from pyspark_utils.performance import avoid_shuffles

df_small = avoid_shuffles.select_before_join(
    df,
    ["policy_number", "claim_amount"]
)


### Remover colunas pesadas antes do join

In [ ]:
df_light = avoid_shuffles.drop_before_join(
    df,
    ["claim_type"]  # coluna string pesada
)


### Repartitionar pelo join key

In [ ]:
df_repart = avoid_shuffles.repartition_on_key(df, "policy_number")


### Coalesce após filtros pesados

In [ ]:
df_filtered = df.filter("claim_amount > 1000")
df_compact = avoid_shuffles.coalesce_after_filter(df_filtered, num_partitions=1)


### Alertas sobre operações que causam wide shuffles

In [ ]:
avoid_shuffles.avoid_wide_transformations(df)

'''
⚠️ Evite estas operações sem necessidade:
- groupBy sem partitioning adequado
- distinct em datasets grandes
- joins sem repartition
- orderBy global
- explode em arrays muito grandes
'''

### Pré‑filtro antes do join

In [ ]:
df_pre_filtered = avoid_shuffles.pre_filter_before_join(
    df,
    "claim_amount > 500"
)
